In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 Extreme XGBoost Classifier with Configurable Class Sample Count (`models/xgboost_feng_esi1_extreme.ipynb`)

This notebook trains a **Binary XGBoost Gradient Boosted Decision Tree Classifier** for **ESI 1 vs Not ESI 1** using **13 Clinical Feature Engineered Predictors** with **Configurable Training Set Sample Regulation**:

### System Architecture & Workflow
1. **Stratified Partitioning First**: Splits the dataset into Train (70%), Validation (15%), and Test (15%) splits before resampling to prevent data leakage.
2. **Configurable Training Sample Count per Class**: Provides explicit knobs (`sample_ratio_not_1` and `target_not_1_count`) on `train_df` to regulate majority class representation (e.g. 1:1, 2:1, 5:1, or exact numeric row caps).
3. **Feature Set (13 Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign anomaly flags (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
4. **Binary XGBoost Gradient Boosting**: Fits binary objective decision trees (`objective = "binary:logistic"`, `eval_metric = "logloss"`) via `xgb.DMatrix` and `xgb.train()`.
5. **Comprehensive Benchmarking Across Splits**: Evaluates Train, Validation, and Test performance (Accuracy, Precision, Recall/Sensitivity, F1 Score, PR-AUC, ROC-AUC).
6. **Reports & Artifacts**:
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_feng_esi1_metrics_barchart.png`).
   - **CSV Reports**: `reports/xgboost_feng_esi1_val_report.csv`, `reports/xgboost_feng_esi1_test_report.csv`.
   - **Model Export**: Saved to `deploy/xgboost_feng_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)
library(xgboost)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 FE Inputs & Complete Case Analysis
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
raw_esi <- as.character(raw_df[[target_col]])
df_feng$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
initial_rows <- nrow(df_feng)
df_feng <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_feng), nrow(df_feng)))
cat(sprintf("Full Complete Case Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_feng), ncol(df_feng)))
cat("Natural Binary Target Distribution ('1' vs 'not_1'):\n")
print(table(df_feng$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning FIRST & Configurable Training Sample Count per Class
# ---------------------------------------------------------
set.seed(config$training$random_state)
# CONFIGURABLE TRAINING SAMPLE COUNT / RATIO PER CLASS
target_not_1_count <- NULL  # Set explicit target row count for 'not_1' (e.g. 5000, 10000, 20000), or NULL for ratio-based
sample_ratio_not_1 <- 4.0   # Ratio of 'not_1' samples relative to ESI 1 count (1.0 = 1:1 balanced, 2.0 = 2:1, 5.0 = 5:1)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
cat(sprintf("Pre-sampling Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))
# ---------------------------------------------------------
# APPLY CONFIGURABLE CLASS COUNT REGULATION TO TRAINING SET ONLY
# ---------------------------------------------------------
idx_1_tr     <- which(train_df$target_layer1 == "1")
idx_not_1_tr <- which(train_df$target_layer1 == "not_1")
n_esi1_tr <- length(idx_1_tr)
if (!is.null(target_not_1_count)) {
  n_not_1_keep <- min(target_not_1_count, length(idx_not_1_tr))
} else {
  n_not_1_keep <- min(as.integer(n_esi1_tr * sample_ratio_not_1), length(idx_not_1_tr))
}
kept_not_1_tr <- sample(idx_not_1_tr, size = n_not_1_keep)
kept_1_tr     <- idx_1_tr
train_df <- train_df[sort(c(kept_1_tr, kept_not_1_tr)), ]
cat(sprintf("Configurable Training Sample Count Applied (Ratio: %.1f | Target Count: %s):\n",
            sample_ratio_not_1, ifelse(is.null(target_not_1_count), "Automatic Ratio", as.character(target_not_1_count))))
cat(sprintf("  - ESI 1 Count:      %d rows\n  - 'not_1' Count:    %d rows\n  - Total Train Size:  %d rows\n\n",
            length(kept_1_tr), length(kept_not_1_tr), nrow(train_df)))
cat("Sample-Regulated Target Distribution in Training Set ('1' vs 'not_1'):\n")
print(table(train_df$target_layer1))
cat("\nNatural Validation Set Target Distribution:\n")
print(table(val_df$target_layer1))
cat("\nNatural Test Set Target Distribution:\n")
print(table(test_df$target_layer1))
feat_names <- setdiff(names(train_df), "target_layer1")
X_train <- as.matrix(train_df[, feat_names])
y_train <- ifelse(train_df$target_layer1 == "1", 1, 0)
X_val   <- as.matrix(val_df[, feat_names])
y_val   <- ifelse(val_df$target_layer1 == "1", 1, 0)
X_test  <- as.matrix(test_df[, feat_names])
y_test  <- ifelse(test_df$target_layer1 == "1", 1, 0)
dtrain_xgb <- xgb.DMatrix(data = X_train, label = y_train)
dval_xgb   <- xgb.DMatrix(data = X_val, label = y_val)
dtest_xgb  <- xgb.DMatrix(data = X_test, label = y_test)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary XGBoost Model on Sample-Regulated Training Data
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training Binary XGBoost Model on Sample-Regulated Training Data (ESI 1 vs Not ESI 1)...\n")
xgb_params <- list(
  objective        = "binary:logistic",
  eval_metric      = "logloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(
  params    = xgb_params,
  data      = dtrain_xgb,
  nrounds   = 150,
  evals     = list(train = dtrain_xgb, val = dval_xgb),
  early_stopping_rounds = 20,
  verbose   = 0
)
cat("XGBoost ESI 1 Binary Classification Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Benchmark Across Splits & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_xgboost_esi1 <- function(model, dmatrix, actual_factor, set_name) {
  prob_1   <- predict(model, newdata = dmatrix)
  pred_val <- ifelse(prob_1 >= 0.5, "1", "not_1")
  pred_fac <- factor(pred_val, levels = c("1", "not_1"))
  act_fac  <- factor(actual_factor, levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  pr_auc  <- calc_pr_auc(ifelse(act_fac == "1", 1, 0), prob_1)
  roc_obj <- tryCatch(pROC::roc(act_fac, prob_1, levels = c("not_1", "1")), error = function(e) NULL)
  roc_auc <- if (!is.null(roc_obj)) as.numeric(roc_obj$auc) else NA
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = c("1", "not_1"),
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(c(prec, ifelse(is.na(cm$byClass["Neg Pred Value"]), 0, cm$byClass["Neg Pred Value"])), 4),
    Recall       = round(c(rec, ifelse(is.na(cm$byClass["Specificity"]), 0, cm$byClass["Specificity"])), 4),
    PR_AUC       = round(c(pr_auc, NA), 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   BINARY ESI 1 XGBOOST (SAMPLE-REGULATED) - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ESI 1 Precision      : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  ESI 1 Recall (Sens)  : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  ESI 1 F1 Score       : %.4f\n", f1))
  cat(sprintf("  ESI 1 PR-AUC         : %.4f\n", pr_auc))
  cat(sprintf("  ROC-AUC              : %.4f\n", roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Performance Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, prec = prec, rec = rec, f1 = f1, pr_auc = pr_auc, roc_auc = roc_auc, prob_1 = prob_1, report_df = report_df))
}
res_train <- evaluate_xgboost_esi1(xgb_model, dtrain_xgb, train_df$target_layer1, "Train")
res_val   <- evaluate_xgboost_esi1(xgb_model, dval_xgb,   val_df$target_layer1,   "Validation")
res_test  <- evaluate_xgboost_esi1(xgb_model, dtest_xgb,  test_df$target_layer1,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "xgboost_feng_esi1_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "xgboost_feng_esi1_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/xgboost_feng_esi1_val_report.csv\n")
cat("Test CSV Report written to:       reports/xgboost_feng_esi1_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split     = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy  = c(res_train$acc,  res_val$acc,  res_test$acc),
  Precision = c(res_train$prec, res_val$prec, res_test$prec),
  Recall    = c(res_train$rec,  res_val$rec,  res_test$rec),
  PR_AUC    = c(res_train$pr_auc, res_val$pr_auc, res_test$pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics Comparison (Binary ESI 1 XGBoost)",
       subtitle = "Comparing Accuracy, Precision, Recall, and PR-AUC across splits",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "xgboost_feng_esi1_metrics_barchart.png"), plot = p_bar, width = 9, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_feng_esi1_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save XGBoost ESI 1 Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_feng_esi1_extreme_model.rds")
saveRDS(list(model = xgb_model, preproc = preproc), file = model_path)
cat("Binary ESI 1 XGBoost model saved to:", model_path, "\n")